In [1]:
from pathlib import Path

import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold, cross_validate

from support_assistant.data import load_intents
from support_assistant.model import build_pipeline, split_intents

y_true = ["card_arrival"] * 6 + ["exchange_rate"] * 3 + ["lost_or_stolen_card"]
y_pred = ["card_arrival"] * 6 + ["exchange_rate", "exchange_rate", "card_arrival"] + ["card_arrival"]

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, zero_division=0))
print("accuracy:", accuracy_score(y_true, y_pred))
print("macro-F1:", round(f1_score(y_true, y_pred, average="macro", zero_division=0), 4))
print("weighted F1:", round(f1_score(y_true, y_pred, average="weighted", zero_division=0), 4))

[[6 0 0]
 [1 2 0]
 [1 0 0]]
                     precision    recall  f1-score   support

       card_arrival       0.75      1.00      0.86         6
      exchange_rate       1.00      0.67      0.80         3
lost_or_stolen_card       0.00      0.00      0.00         1

           accuracy                           0.80        10
          macro avg       0.58      0.56      0.55        10
       weighted avg       0.75      0.80      0.75        10

accuracy: 0.8
macro-F1: 0.5524
weighted F1: 0.7543


In [3]:
ROOT = Path.cwd().parents[2]
df = load_intents(ROOT / "data" / "raw" / "banking77_train.csv")
train_df, val_df = split_intents(df)

pipeline = build_pipeline()
pipeline.fit(train_df["text"], train_df["category"])
predicted = pipeline.predict(val_df["text"])
y_val = val_df["category"]

print("accuracy:", round(accuracy_score(y_val, predicted), 4))
print("macro-F1:", round(f1_score(y_val, predicted, average="macro", zero_division=0), 4))

report = pd.DataFrame(classification_report(y_val, predicted, output_dict=True, zero_division=0)).T
per_intent = report.iloc[:-3].sort_values("f1-score")
per_intent.head(8).round(2)

accuracy: 0.8605
macro-F1: 0.8583


,precision,recall,f1-score,support
virtual_card_not_working,0.75,0.38,0.50,8.0
why_verify_identity,0.83,0.62,0.71,24.0
transfer_not_received_by_recipient,0.68,0.76,0.72,34.0
card_acceptance,0.80,0.67,0.73,12.0
supported_cards_and_currencies,0.68,0.81,0.74,26.0
top_up_failed,0.73,0.76,0.75,29.0
card_delivery_estimate,0.69,0.82,0.75,22.0
card_not_working,0.69,0.82,0.75,22.0


In [7]:
report = pd.DataFrame(classification_report(y_val, predicted, output_dict=True, zero_division=0)).T
per_intent = report.sort_values("f1-score")
per_intent.head(8).round(2)

,precision,recall,f1-score,support
virtual_card_not_working,0.75,0.38,0.50,8.0
why_verify_identity,0.83,0.62,0.71,24.0
transfer_not_received_by_recipient,0.68,0.76,0.72,34.0
card_acceptance,0.80,0.67,0.73,12.0
supported_cards_and_currencies,0.68,0.81,0.74,26.0
top_up_failed,0.73,0.76,0.75,29.0
card_delivery_estimate,0.69,0.82,0.75,22.0
card_not_working,0.69,0.82,0.75,22.0


In [8]:
for seed in range(5):
    seed_train, seed_val = split_intents(df, seed=seed)
    model = build_pipeline().fit(seed_train["text"], seed_train["category"])
    seed_predicted = model.predict(seed_val["text"])
    macro = f1_score(seed_val["category"], seed_predicted, average="macro", zero_division=0)
    print("seed", seed, "| accuracy", round(accuracy_score(seed_val["category"], seed_predicted), 4), "| macro-F1", round(macro, 4))

seed 0 | accuracy 0.8525 | macro-F1 0.8453
seed 1 | accuracy 0.851 | macro-F1 0.8459
seed 2 | accuracy 0.857 | macro-F1 0.855
seed 3 | accuracy 0.8605 | macro-F1 0.8537
seed 4 | accuracy 0.86 | macro-F1 0.8561


In [9]:
unique = df[~df["text"].str.lower().str.split().str.join(" ").duplicated()]
splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

result = cross_validate(
    build_pipeline(), unique["text"], unique["category"], cv=splitter, scoring=["accuracy", "f1_macro"]
)
print(sorted(result))
print("macro-F1 per fold:", result["test_f1_macro"].round(4))
print("macro-F1:", round(result["test_f1_macro"].mean(), 4), "±", round(result["test_f1_macro"].std(), 4))
print("accuracy:", round(result["test_accuracy"].mean(), 4), "±", round(result["test_accuracy"].std(), 4))

['fit_time', 'score_time', 'test_accuracy', 'test_f1_macro']
macro-F1 per fold: [0.8351 0.8588 0.8656 0.8586 0.8513]
macro-F1: 0.8539 ± 0.0104
accuracy: 0.8591 ± 0.0084
